In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import notebook_login
from sae_lens import SAE

notebook_login()
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [2]:
ds = load_dataset("openai/gsm8k", "main")
train_ds = ds["train"]
test_ds = ds["test"]

In [40]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [58]:
v_steer = torch.load('steering_vector_num_acc_val.pt')
v_steer_random = torch.randn(4096)
v_steer_random *= v_steer.abs().sum() / v_steer_random.abs().sum()

In [79]:
layer_idx = 19
target_module = model.model.layers[19]

In [80]:
def make_hook(vec, coeff=1.0, token_position="all"):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hidden = output[0]
        else:
            hidden = output

        # Normalize the steering direction to unit length
        direction = vec / torch.linalg.norm(vec)
        direction = direction.to(device=hidden.device, dtype=hidden.dtype)

        # hidden shape: [batch, seq, hidden_size]
        if token_position == "all":
            # Project out existing component along direction, then set to coeff
            proj = torch.sum(hidden * direction, dim=-1, keepdim=True)  # [batch, seq, 1]
            hidden = hidden.clone()
            hidden = hidden - proj * direction + coeff * direction
        elif token_position == "last":
            hidden = hidden.clone()
            proj = torch.sum(hidden[:, -1, :] * direction, dim=-1, keepdim=True)  # [batch, 1]
            hidden[:, -1, :] = hidden[:, -1, :] - proj * direction + coeff * direction
        else:
            raise ValueError("token_position must be 'last' or 'all'")

        if isinstance(output, tuple):
            return (hidden,) + output[1:]
        return hidden
    return hook

In [82]:
prompt = ds['train'][2]['question']
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
prompt

'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?'

In [84]:
torch.linalg.norm(v_steer_random)

tensor(0.3375)

In [88]:
coeff=15
with torch.no_grad():
    random_handle = target_module.register_forward_hook(make_hook(v_steer_random, coeff=coeff, token_position="all"))
    steered_out = model.generate(**inputs, max_new_tokens=200)
    print('RANDOMLY STEERED OUTPUT:')
    print(tokenizer.decode(steered_out[0], skip_special_tokens=True))
    random_handle.remove()

    handle = target_module.register_forward_hook(make_hook(v_steer, coeff=coeff, token_position="all"))
    steered_out = model.generate(**inputs, max_new_tokens=200)
    print('STEERED OUTPUT:')
    print(tokenizer.decode(steered_out[0], skip_special_tokens=True))
    handle.remove()

    out = model.generate(**inputs, max_new_tokens=200)
    print('UNSTEERED OUTPUT:')
    print(tokenizer.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


RANDOMLY STEERED OUTPUT:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?**

Okay, let me try to figure out this problem step by step. So, Betty wants to buy a new wallet that costs $100. She's already saving money, but she's only got half of the money she needs. Hmm, so first, let's figure out how much money Betty has saved so far.

If the total cost is $100 and she's only half of that, that means Betty has saved $50. Right? Because half of 100 is 50. So, Betty has $50 so far.

Now, Betty's parents are going to give her some money for this. The problem says her parents are giving her $15. That's straightforward. So, Betty's parents are giving her $15.

Then, her grandparents are going to give her twice as much as her parents. Hmm, so if her parents give $15, then the grandpare

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


STEERED OUTPUT:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet? Wait, that seems conflicting.

Wait, maybe I need to clarify: If the wallet is $100, and Betty has the opposite. Let me re-express.

Wait, let me read again: "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wait, that seems conflicting.

Wait, no, it's the opposite. Wait, the wording is: Betty is saving money for a new wallet which costs $100. Wait, actually, no. Wait, hold on, the wording is: "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided

In [81]:
for name, m in model.named_modules():
    hooks = (
        len(m._forward_hooks)
        + len(m._forward_pre_hooks)
        + len(m._backward_hooks)
    )
    if hooks:
        print(name, hooks)

In [28]:
inputs = tokenizer('a', return_tensors="pt").to(model.device)
handle = target_module.register_forward_hook(make_hook(v_steer, coeff=1.0, token_position="all"))
steered_out = model.generate(**inputs, max_new_tokens=3)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated
Hook activated


In [29]:
steered_out

tensor([[   64, 10134,  3604,   358]], device='cuda:0')

In [ ]:
## OLD CODE: NAIVE ADDITION

def make_hook(vec, coeff=1.0, token_position="last"):
    '''
    Wrapper class to make a hook function, called during the forward pass of the
    '''
    def hook(module, inputs, output):
        # Many HF modules return either:
        #   - a tensor
        #   - a tuple whose first element is the hidden states
        if isinstance(output, tuple):
            hidden = output[0]
        else:
            hidden = output

        vec_cast = (coeff * vec).to(device=hidden.device, dtype=hidden.dtype)

        # hidden shape is usually [batch, seq, hidden_size]
        if token_position == "last":
            hidden = hidden.clone()
            hidden[:, -1, :] = hidden[:, -1, :] + vec_cast
        elif token_position == "all":
            hidden = hidden + vec_cast.view(1, 1, -1)
        else:
            raise ValueError("token_position must be 'last' or 'all'")

        if isinstance(output, tuple):
            return (hidden,) + output[1:]
        return hidden
    return hook

In [22]:
feature_idx = 46379
steering_vec = sae.W_dec[feature_idx].clone()

In [16]:
def get_max_activation(model, tokenizer, sae, feature_idx, prompts, device, hook_layer=19):
    max_act = 0.0
    hook_layer = model.model.layers[hook_layer]

    def capture_hook(module, input, output):
        nonlocal max_act
        hidden = output[0] if isinstance(output, tuple) else output
        hidden_cast = hidden.to(device=sae.device, dtype=sae.W_enc.dtype)
        features = sae.encode(hidden_cast)
        peak = features[..., feature_idx].abs().max().item()
        max_act = max(max_act, peak)

    handle = hook_layer.register_forward_hook(capture_hook)
    try:
        for prompt in prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                model(**inputs)
    finally:
        handle.remove()

    return max_act

In [3]:
from typing import List, Tuple, Callable
from torch import Tensor

def add_hooks(
    module_forward_pre_hooks: List[Tuple[torch.nn.Module, Callable]],
    module_forward_hooks: List[Tuple[torch.nn.Module, Callable]],
    **kwargs
):
    """
    Context manager for temporarily adding forward hooks to a model.

    Parameters
    ----------
    module_forward_pre_hooks
        A list of pairs: (module, fnc) The function will be registered as a
            forward pre hook on the module
    module_forward_hooks
        A list of pairs: (module, fnc) The function will be registered as a
            forward hook on the module
    """
    try:
        handles = []
        for module, hook in module_forward_pre_hooks:
            partial_hook = functools.partial(hook, **kwargs)
            handles.append(module.register_forward_pre_hook(partial_hook))
        for module, hook in module_forward_hooks:
            partial_hook = functools.partial(hook, **kwargs)
            handles.append(module.register_forward_hook(partial_hook))
        yield
    finally:
        for h in handles:
            h.remove()

def get_clamp_hook(
    direction: Tensor,
    max_activation: float = 1.0,
    strength: float = 1.0
):
    def hook_fn(module, input, output):
        nonlocal direction
        if torch.is_tensor(output):
            activations = output.clone()
        else:
            activations = output[0].clone()
        
        direction = direction / torch.norm(direction)
        direction = direction.type_as(activations)
        proj_magnitude = torch.sum(activations * direction, dim=-1, keepdim=True)
        orthogonal_component = activations - proj_magnitude * direction

        clamped = orthogonal_component + direction * max_activation * strength

        if torch.is_tensor(output):
            return clamped
        else:
            return (clamped,) + output[1:] if len(output) > 1 else (clamped,)

    return hook_fn

In [4]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
device = "cuda"

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.eval()

sae, _, _ = SAE.from_pretrained(
    release="andreuka18/deepseek-r1-distill-llama-8b-lmsys-openthoughts",  # check sae_lens for exact release name
    sae_id="blocks.19.hook_resid_post"               # layer 19, residual stream post
).to(device=device, dtype=model.dtype)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/cstam5916/pyenvs/cfd/lib/python3.11/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(
/tmp/ipykernel_1093837/203719932.py:8: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, _, _ = SAE.from_pretrained(


In [11]:
ds = load_dataset("openai/gsm8k", "main")

In [37]:
feature_idx = 48026
steering_vec = sae.W_dec[feature_idx].clone()
max_activation = get_max_activation(model, tokenizer, sae, feature_idx, 
                                     list(ds['train']['question'])[:100],
                                     device)
strength = 2.0

In [38]:
hook_layer = model.model.layers[19]
hook_fn = get_clamp_hook(steering_vec, max_activation=max_activation, strength=strength)

In [29]:
ds['test']['question'][3]

'James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?'

In [35]:
# --- generate WITHOUT steering ---
prompt = 'Prove Fermat\'s Little Theorem.'
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out_plain = model.generate(**inputs, max_new_tokens=500)
print("=== NO STEERING ===")
print(tokenizer.decode(out_plain[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


=== NO STEERING ===
Prove Fermat's Little Theorem. Hmm, I remember that Fermat's Little Theorem states that if p is a prime number, then for any two integers a and b, a^p ≡ a mod p. But I need to prove it. Let me think about how to approach this.

First, let me recall the theorem again: If p is a prime and a is an integer not divisible by p, then a^(p-1) ≡ 1 mod p. This is another version of the theorem. So maybe I can prove this version and then see if it can lead to the original statement.

Alternatively, the original theorem says that a^p ≡ a mod p for any integer a. So if I can show that a^p ≡ a mod p, regardless of whether a is divisible by p or not, that would be sufficient.

I think I need to use some properties of modular arithmetic and perhaps group theory. Wait, Fermat's Little Theorem is related to the order of an element in a group. If I consider the multiplicative group of integers modulo p, which is a group under multiplication modulo p, then the order of an element a in 

In [ ]:
# --- generate WITH steering ---
handle = hook_layer.register_forward_hook(hook_fn)
try:
    with torch.no_grad():
        out_steered = model.generate(**inputs, max_new_tokens=500)
finally:
    handle.remove()

print("=== STEERED ===")
print(tokenizer.decode(out_steered[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


=== STEERED ===
Prove Fermat's Little Theorem. Hmm, okay, so Fermat's Little Theorem says that if p is a prime number, then for any two integers a and b, a^b mod p is congruent to a^(b mod (p-1)) mod p. Wait, no, actually, more precisely, it says that if p is prime, then for any integer a not divisible by p, a^(p-1) ≡ 1 mod p. And also, a^b ≡ a^(b mod (p-1)) mod p. Hmm, or is it that a^b ≡ a^(b mod (p-1)) mod p? Let me check.

Actually, I think Fermat's Little Theorem has two parts. The first part is that if p is prime and a is not divisible by p, then a^(p-1) ≡ 1 mod p. The second part is that a^b ≡ a^(b mod (p-1)) mod p, but I think this is only under certain conditions.

Wait, maybe it's better to look at the exact statement. Fermat's Little Theorem: If p is a prime number, then for any integer a, the number a^p is congruent to a mod p. That is, a^p ≡ a mod p. So that's one part.

And the other part is that if a and p are coprime, then a^(p-1) ≡ 1 mod p. So, perhaps, the theorem has

: 